# SDY2823归一化复评：SSD重算记录

## tl;dr

2026-10-13，固定96位供者、模型权重、原始查询和流式评价标签，完成384次完整portrait推断。本轮增加10种条件，连同17种旧条件共复核243个问题—改写—方法指标组。

主条件完整portrait的平均nDCG@10为0.203，并列分数期望为0.198；EPIC为0.800。RNA直接检索为0.327，本次基因打乱对照为0.411。

**本轮没有显示预期的独立细胞状态检索增益，不支持据此进入Cell确认性研究或宣称模型具有检索优势。** 这是3个相关问题在单一队列中的探索性结果，不能推出模型在所有生物学任务上均无效。

## Context & Methods

原工作目录迁移期间，SSD副本没有纳入刚完成的推断输出。作者明确授权在SSD重算；本轮没有读取或恢复废纸篓。原计算曾报告成功，但其文件未能重新核验，本报告只采用当前SSD重算的可验证结果。

### Key Assumptions

- 主条件在23,323个已映射实测基因上估计队列size factors；敏感性条件在58,302个实测Ensembl行上估计后再映射。两者均只用同一96个候选样本。
- 这是利用整个候选队列估计参数的无监督归一化，不是仅由训练集确定的参考，也不是单样本部署协议。
- 30题各3种改写均已打分；只有Q01、Q02、Q19具备本轮T/B流式相关性标签。其余27题不能计为已经验证。
- 每题先平均3种改写，再对3题取均值；不把9个改写当作9个独立生物学实验，也不给一般性优越性的P值。
- 所有17个旧方法及其原始预处理保留。EPIC/MCP-counter使用固定的查询到分子特征映射，非通用自由文本解析；该人工映射成本没有测量。
- 两个RNA分支接收同一预处理输入；禁用第二次log/CPM和结构化分支的备用归一化。未重训、未调整阈值或依据结果选择归一化。

## Data

### 1. 校验输入、输出和指标

父目录PROTOCOL.json与preparation.json锁定原始输入。execution_contract.json锁定代码、权重、依赖和查询；execution_summary.json与evaluation_summary.json分别索引推断输出和评价表。verify_outputs.py独立重算全部900组新得分矩阵以及243组评价指标。

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from verify_outputs import verify
out = Path.cwd()
input_root = out.parent
study_root = input_root.parent
checks = verify(study_root, input_root, out)
assert checks == json.loads((out/'independent_validation.json').read_text())
summary = json.loads((out/'evaluation_summary.json').read_text())
assert summary['previous_153_metric_groups_maximum_difference'] < 1e-12
print(f"900组得分矩阵重构误差 {checks['score_max_abs_error']:.1f}；243组独立指标最大误差 {checks['independent_metric_max_abs_error']:.3g}；384份推断及输入路径验证通过。")
print('旧153组指标仅存在浮点求和量级差异，原始文件哈希未变。')

900组得分矩阵重构误差 0.0；243组独立指标最大误差 3.33e-16；384份推断及输入路径验证通过。
旧153组指标仅存在浮点求和量级差异，原始文件哈希未变。


## Results

### 2. 主条件与预定对照

nDCG的范围为0–1，越高表示前10名中更相关的样本排序越靠前。高等级候选数为每10个返回样本中达到固定流式上四分位等级的平均数量，不是人工认可的statement数量。

In [2]:
means = pd.read_csv(out/'three_question_descriptive_means.csv', float_precision='round_trip').set_index('method')
pd.set_option('display.max_colwidth',None)
pd.set_option('display.max_rows',40)
selected = {
 'EPIC_BRef_raw':'EPIC（固定查询映射）',
 'cellwhisperer_v1':'CellWhisperer',
 'full_portrait_text_released_calibration':'完整portrait：旧CPM',
 'full_portrait_released_calibration_sf_reference':'完整portrait：主条件',
 'rna_direct_sf_reference':'RNA直接空间：主条件',
 'rna_fused_sf_reference':'RNA融合空间：主条件',
 'rna_value_shuffled_sf_reference':'主条件：基因值打乱',
 'rna_mismatched_sf_reference':'主条件：样本错配'}
view = means.loc[list(selected), ['ndcg_at_k','tie_expected_ndcg_at_10','strong_hits_per_10']].rename(index=selected,columns={'ndcg_at_k':'nDCG@10','tie_expected_ndcg_at_10':'并列期望nDCG@10','strong_hits_per_10':'高等级候选数/10'})
view.round(3)

,nDCG@10,并列期望nDCG@10,高等级候选数/10
method,,,
EPIC（固定查询映射）,0.800,0.800,6.667
CellWhisperer,0.337,0.337,2.556
完整portrait：旧CPM,0.395,0.338,3.222
完整portrait：主条件,0.203,0.198,1.444
RNA直接空间：主条件,0.327,0.327,2.556
RNA融合空间：主条件,0.373,0.373,3.000
主条件：基因值打乱,0.411,0.411,3.333
主条件：样本错配,0.239,0.239,1.889


### 3. 分问题检查

Q01为T细胞，Q02为B细胞，Q19为二者组合。三题有生物学相关性。以下保留预定主条件和对照，不因均值不利而删除个别问题。

In [3]:
per_question = pd.read_csv(out/'per_question_results.csv')
comparison = ['EPIC_BRef_raw','cellwhisperer_v1','full_portrait_released_calibration_sf_reference','rna_direct_sf_reference','rna_value_shuffled_sf_reference']
table = per_question.loc[per_question.method.isin(comparison)].pivot(index='method',columns='query_id',values='ndcg_at_k').loc[comparison]
table.rename(index=selected).round(3)

query_id,Q01,Q02,Q19
method,,,
EPIC（固定查询映射）,0.754,0.948,0.698
CellWhisperer,0.403,0.372,0.236
完整portrait：主条件,0.157,0.287,0.164
RNA直接空间：主条件,0.281,0.452,0.249
主条件：基因值打乱,0.521,0.359,0.354


### 4. 全部27个条件

两种基因全集条件的完整portrait指标相同；校准开关在各条件中也未改变96份被检索的文字。它们不是独立重复实验。完整文字由旧CPM的23种增加到56种，并不代表检索相关性或语句正确率提高。

In [4]:
assert len(means)==27
assert set(summary['full_portrait_unique_texts_by_condition'].values()) == {56}
assert set(summary['same_text_with_and_without_calibration'].values()) == {96}
means[['ndcg_at_k','tie_expected_ndcg_at_10','precision_at_k','paraphrase_top10_jaccard']].rename(columns={'ndcg_at_k':'nDCG@10','tie_expected_ndcg_at_10':'并列期望','precision_at_k':'Precision@10','paraphrase_top10_jaccard':'改写Jaccard'}).round(3)

,nDCG@10,并列期望,Precision@10,改写Jaccard
method,,,,
EPIC_BRef_TB_fixed_text_minilm,0.391,0.347,0.578,0.556
EPIC_BRef_known_leukocytes,0.800,0.800,0.933,1.000
EPIC_BRef_raw,0.800,0.800,0.933,1.000
MCPcounter_TB_fixed_text_minilm,0.344,0.247,0.422,0.556
MCPcounter_log2cpm,0.564,0.563,0.867,1.000
MCPcounter_log2tpm,0.578,0.577,0.833,1.000
cellwhisperer_v1,0.337,0.337,0.444,0.261
full_portrait_no_calibration_sf_full_measured,0.203,0.198,0.456,0.300
full_portrait_no_calibration_sf_reference,0.203,0.198,0.456,0.300


### 5. 输入完整性和诊断边界

每个样本匹配3,524/4,096个选定模型基因，未解析的572个仍按冻结模型的零输入约定处理；没有将缺失解释为生物学零。以下是实测特征超出训练标准化后±8裁切范围的比例中位数。没有丢弃表现不佳或覆盖不足的样本。

In [5]:
diagnostics = pd.read_csv(out/'input_diagnostics.csv')
assert len(diagnostics)==192 and diagnostics.matched_genes.eq(3524).all()
assert diagnostics.normalization_library_sum.isna().all()
print('96位供者全部保留；两种条件各96；完整推断失败0；没有第二次CPM转换。')
diagnostics.groupby('arm')[['matched_feature_upper_clip_fraction','matched_feature_lower_clip_fraction']].median().rename(columns={'matched_feature_upper_clip_fraction':'上限裁切比例中位数','matched_feature_lower_clip_fraction':'下限裁切比例中位数'}).round(6)

96位供者全部保留；两种条件各96；完整推断失败0；没有第二次CPM转换。


,上限裁切比例中位数,下限裁切比例中位数
arm,,
sf_full_measured,0.0,0.0
sf_reference,0.0,0.0


## Takeaways

- 计算与数据连接通过核查：全部输入/输出哈希、384份逐样本记录、900组得分和243组指标可追溯，没有候选丢失或重复连接。
- 科学推进门槛未通过：恢复历史归一化公式没有建立相对强传统基线的状态检索增益；基因打乱对照也未出现预期的性能下降。单次打乱不能估计随机对照分布，更不能解释为模型鲁棒。
- 这不能单独证明失败的根因。模型训练目标、跨平台差异、未解析基因和细胞组成任务的匹配程度仍需有界诊断；不要在已看过结果的3题上反复调参后当作确认性证据。
- 按原计划，先保留阴性结果并重新评估研究定位，不扩大确认性或湿实验投入，不把格式完成写成Cell投稿准备完成。
- 全部过程保存在当前SSD目录；主稿v1.12、提交原件、历史结果和权重不变。独立人工语句盲评、anchor训练划分来源、单样本部署参考和新留出队列的实用增益仍未解决。

本报告的质量结论是“可作为有明确边界的探索性阴性结果分享”，不是“已达到Cell投稿科学门槛”。时间为单次推断日志，不用于跨方法速度结论。没有新的时间序列数据，故不作时间趋势分析。